In [11]:
import re

# List of strings to be searched
data = """
cond-mat0003325.html
hep-ph0003287.html
gr-qc0003030.html
quant-ph_0004105
0001.2312
"""

# Regular expression to find the first occurrence of any four digits


# Split the data into individual lines
lines = data.strip().split('\n')

# Loop through each line
for line in lines:
    # Search for the pattern in the line
    match = re.search(r"\d{4}", line)
    if match:
        # Print the matched pattern
        print(f"Found '{match.group()}' in: {line}")

Found '0003' in: cond-mat0003325.html
Found '0003' in: hep-ph0003287.html
Found '0003' in: gr-qc0003030.html
Found '0004' in: quant-ph_0004105
Found '0001' in: 0001.2312


In [1]:
%cd ../

from uparxive.xml_to_json.html_to_dense_text import *

from tqdm.auto import tqdm
import os
verbose = False
filte_out_note = True
reterive_result_mode = False
use_count_type_ref= False

from bs4 import BeautifulSoup

/nas/zhangtianning.di/projects/unique_data_build


# function

In [ ]:
###############
"""
Some known problem:
    - ~~See 0709.2524: The section after \appendix will not be collected into the main content. (The latexml do generate appendix, so we should fix it here)~~

"""
import re
from lxml import etree
from typing import Dict
from copy import deepcopy
import lxml,copy
import logging
import os
from uparxive.xml_to_json.check_string_is_citation import *
from uparxive.batch_run_utils import BatchModeConfig, dataclass
from tqdm.auto import tqdm
### set the loger in warning mode
log_level = os.environ.get('LOG_LEVEL', 'WARN')
logging.basicConfig(level=log_level, format='%(asctime)s - %(levelname)s - %(message)s')

prepositions = {'at', 'in', 'on', 'for', 'with', 'and','see', 'or', 'nor', 'about', 'as', 'by', 'over', 'according to', 'against', 'along', 'among', 'apart from', 'around', 'as for', 'aside from', 'because of', 'before', 'behind', 'below', 'beneath', 'beside', 'between', 'beyond', 'but', 'by means of', 'concerning', 'despite', 'down', 'due to', 'during', 'except', 'except for', 'in addition to', 'in case of', 'in front of', 'in place of', 'in spite of', 'inside', 'instead of', 'into', 'like', 'near', 'next', 'off', 'onto', 'out', 'out of', 'outside', 'over', 'past', 'since', 'through', 'throughout', 'toward', 'under', 'underneath', 'until', 'up', 'upon', 'with', 'within', 'without'}
PATTERN_END_REF = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['ref', 'refs', 'papers', 'paper','reference','references']])
PATTERN_END_FIG = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['fig', 'figures', 'figs', 'figure']])
PATTERN_END_TAB = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['tab', 'tables', 'tabs', 'table']])
PATTERN_END_SEC = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['sec', 'section']])
PATTERN_END_EQU = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['eq', 'equ', 'equation', 'equations','formula','formulas']])
PATTERN_END_CHP = "|".join(["(?<=\s|\(|\{|\[)"+k+"$|^"+k+"$" for k in ['chapter','chapters','chpt','chpts']])


enable_checkCiteDontHaveBibRef = False
enable_checkTooManyNote= True
class KnowError(NotImplementedError):pass
class MathDontHaveTex(NotImplementedError):pass
class MisMatchRefError(NotImplementedError):pass
class CiteDontHaveBibRefError(NotImplementedError):pass

from bs4 import BeautifulSoup
import traceback
@dataclass
class HTMLtoJsonConfig(BatchModeConfig):
    task_name = 'html_to_json'
    reterive_result_mode : bool = False
    passManyNote : bool = False
    passNote : bool = False
    use_origin_ref_number : bool = False
    verbose: bool = False
    

In [764]:
#### functions
from uparxive.xml_to_json.xml_to_dense_text import *
from typing import List, Dict,Tuple
from bs4 import NavigableString

def discard_note(soup):
    for element in soup.find_all('ltx_note'):
        raise NotImplementedError
        remove_a_tagblock(element, 'tags')
        text = '[[[Notice: '
        append_tex = ']]] ' + (element.get_text(strip=True) or "")
        # Insert the text before the 'note' element
        element.insert_before(text)

        # Append the modified content after the 'note' element
        element.insert_after(append_tex)

        # Unwrap the 'note' element, keeping its children
        element.unwrap()
    return soup

def retrieve_all_cite(soup):
    """
    Retrieve all the citation in the soup
        - Type 1: bib ==> <a class="ltx_ref" href="#bib.bib1" title="">1</a>
        - Type 2: fig ==> <a class="ltx_ref" href="#S2.F2" title="Figure 2 ‣ 2 THE QUADRUPOLE TRANSITION TO THE Δ⁢(1232) ‣ Probing the Structure of Nucleons in the Resonance region with CLAS at Jefferson Lab"><span class="ltx_text ltx_ref_tag">2</span></a>,
        - Type 3: tab ==> [TODO]: please give a check
        - Type 3: math==> [TODO]: please give a check
    """
    ref_count = {}
    for ref in soup.find_all(class_='ltx_ref'):
        if ref['href']:
            ref_text = ref['href'].strip().lstrip('#')
            ref_count[ref_text] = ref_count.get(ref_text, 0) + 1

    return ref_count


def identify_bibblock_is_note_or_citation(bibblock:BeautifulSoup, args:HTMLtoJsonConfig)->Tuple[bool,bool]:
    iscitationQ = True 
    hardcitationQ = False
    #for bibblock in bibblocks:
    ref_in_bib = bibblock.find(class_='ltx_bibblock')
    if ref_in_bib is not None:
        if ref_in_bib.get('href',''):
            iscitationQ   = False
            hardcitationQ = True
    
    cite_in_bib = bibblock.find(class_='ltx_ref')
    if cite_in_bib is not None:
        raise NotImplementedError(f"Lets have a look")
        
    for math_in_bib in bibblock.find_all(class_='ltx_Math'):
        raise NotImplementedError(f"Lets have a look")
    
    return iscitationQ, hardcitationQ
                
def parse_bibitem(bibitem: BeautifulSoup, bibindex:int, 
                  note_ref_labels:Dict[str, str], 
                  note_ref_metadata:Dict[str, str], 
                  bibitem_ref_labels:Dict[str, str], 
                  bibitem_ref_metadata:Dict[str, str], 
                  args:HTMLtoJsonConfig):
    """
        <li id="bib.bib1" class="ltx_bibitem">
            <span class="ltx_tag ltx_tag_bibitem">[1]</span>
            <span class="ltx_bibblock"> N. Isgur and G. Karl; Phys. Lett. B72:109 (1977), Phys. Rev. D23, 817 (1981)
            </span>
        </li>
        or
        <li id="bib.bib1" class="ltx_bibitem">
            <span class="ltx_tag ltx_role_refnum ltx_tag_bibitem">[1]</span>
            <span class="ltx_bibblock">
                S. Bansal, J. Read, B. Pourbohloul, and L. A. Meyers.

            </span>
            <span class="ltx_bibblock">The dynamic nature of contact networks in infectious disease
                epidemiology.

            </span>
            <span class="ltx_bibblock"><span id="bib.bib1.1.1" class="ltx_text ltx_font_italic">J. Biol.
                    Dyn.</span>, 4:478–489, 2010.

            </span>
        </li>
    """

    #### Identify the ref_key and the tag of the bibitem
    filte_out_note = args.filte_out_note
    verbose = args.verbose
    tag = bibitem.find(class_='ltx_tag')
    if tag is None:
        logging.warning(f"empty ref ??? ==> {bibitem}")
        return ### 
    if tag is None:
        logging.warning(f"WARNING: the bibitem {bibitem} has no tag")
        refnum_tag = None
    else:
        refnum_tag = tag.text
    
    label_of_bib = bibitem['id']
    if not label_of_bib:
        logging.info(f" this bibitem={bibitem} dont have label ???? ")
        return
    if tag is not None: tag.decompose()
    
    #### now we will analysis whether the content of the bib is a note or citation
    #print(bibitem)
    bibblocks= bibitem.find_all(class_='ltx_bibblock')
    #assert len(bibblocks)==1, f"why this reference string ==> {bibitem} <== has more then one bibblocks ==>{bibblocks}"
    iscitationQ = True
    hardcitationQ=True
    bibstring = better_latex_sentense_string(" ".join([bibblock.text for bibblock in bibblocks]))

    if filte_out_note:
        for bibblock in bibblocks:
            iscitationQ, hardcitationQ = identify_bibblock_is_note_or_citation(bibblock,args)
            break

        if iscitationQ:
            iscitationQ = not should_the_string_be_regard_as_note(bibstring)
            if verbose and not iscitationQ:
                print(f'{bibblocks} is regard as note since it has `string judge`')
        
        if refnum_tag is None:
            refnumtext = f"ref_{bibindex}"
        else:
            refnumtext = refnum_tag # In quant-ph_0102079: it may be <tag role="refnum"><text fontsize="90%">(40)</text></tag> like 
    
    if not iscitationQ:
        ## then, this block is a note, should save whole xml code in this block and put them into main content
        note_ref_labels[label_of_bib]  =  refnumtext
        note_ref_metadata[label_of_bib]= [hardcitationQ, deepcopy(bibitem)]
    else:
        #refnum_int = int(refnumtext)
        bibitem_ref_labels[label_of_bib]  = refnumtext
        bibitem_ref_metadata[label_of_bib]=bibstring

def parse_bibentry(bibitem,bibindex,filte_out_note,note_ref_labels,note_ref_metadata, bibitem_ref_labels,bibitem_ref_metadata,verbose):
    """
    Usually caused by directly write .bib format in .tex file. For example, arxiv: 1004.4054
    """
    raise NotImplementedError
    ### first, find the ref id
    label_of_bib = bibitem['id']
    if label_of_bib is None or len(label_of_bib.strip())==0:
        logging.info(f" this bibitem={bibitem} dont have label ???? ")
        return
    ### bibentry wont have the refnum tag, so lets use the id  
    refnumtext = label_of_bib
    
    ### if we use this, it must be a citation
    bib_origin = bibitem.find('.//default:bib-data[@role="self"]', ns)
    if bib_origin is not None:
        bib_origin.getparent().remove(bib_origin)
        # from python_script.CitationStyleLanguage import CitationStyleLanguage
        # import python_script.bibjson as bibjson
        # bibstring = " ".join(bibstring.itertext())
        # #bibstring = merge_author(bibstring)
        # bibstring = better_latex_sentense_string(bibstring)
        # bibjson_collection = bibjson.collection_from_bibtex_str(bibstring,collection='.bib')
        # if len(bibjson_collection['records']) == 0:
        #     print(bibstring)
        #     print(bibjson_collection)
        #     
        # bibpool   = bibjson_collection['records'][0]
        # citation  = CitationStyleLanguage.from_dict(bibpool)
        # bibstring = citation.to_citation(size='full')
    
    # name  = bibitem.find('.//default:bib-name', ns)
    # title = bibitem.find('.//default:bib-title', ns)
    # type  = bibitem.find('.//default:bib-type', ns)
    # date  = bibitem.find('.//default:bib-date', ns)
    # organization = bibitem.find('.//default:bib-organization', ns)
    # note  = bibitem.find('.//default:bib-note', ns)
    # publisher = bibitem.find('.//default:bib-publisher', ns)
    # volumn= bibitem.find('.//default:bib-part[@role="volume"]', ns)
    # number= bibitem.find('.//default:bib-part[@role="number"]', ns)
    # pages = bibitem.find('.//default:bib-part[@role="pages"]', ns)
    # journel= bibitem.find('.//default:bib-related[@role="host"]', ns)
    bibstring = []
    for child in bibitem:
        bibstring.append(better_latex_sentense_string(" ".join(child.itertext())))
    bibstring = ", ".join(bibstring)
        
    
    bibitem_ref_labels[labels]= refnumtext
    bibitem_ref_metadata[labels]=bibstring

def remove_entire_bibliography_and_build_labels(soup: BeautifulSoup ,args:HTMLtoJsonConfig):

    bibitem_ref_labels = {}
    bibitem_ref_metadata = {}
    note_ref_labels = {}
    note_ref_metadata={}
    for bio_element in soup.find_all(class_='ltx_bibliography'):
        for bio_ul in bio_element.find_all(class_='ltx_biblist'):
            for bibindex, bio_li in enumerate(bio_ul.find_all(class_='ltx_bibitem')):
                parse_bibitem(bio_li,bibindex,note_ref_labels,note_ref_metadata, bibitem_ref_labels,bibitem_ref_metadata,args)
            for bibindex, bio_li in enumerate(bio_ul.find_all(class_='ltx_bibentry')):
                parse_bibentry(bio_li,bibindex,note_ref_labels,note_ref_metadata, bibitem_ref_labels,bibitem_ref_metadata,args)
        bio_element.decompose()
    return soup, bibitem_ref_labels, bibitem_ref_metadata, note_ref_labels, note_ref_metadata

def put_note_string_back_into_each_sentence(soup: BeautifulSoup, 
                                            ref_count: Dict[str, int], 
                                            note_ref_metadata: Dict[str, BeautifulSoup]):
    """
    When put note back into the main content, we always think the <ref> must be in <cite>
    """
    put_back_keys  = set()
    for cite in soup.find_all(class_='ltx_cite'):
        all_refs_of_one_cite = cite.find_all(class_='ltx_ref')
        for bibref in all_refs_of_one_cite:
            refs = bibref.get('href', "").split(',') ### multi_ref must be aaa,bbb,ccc
            assert len(refs)==1, "why a note ref have multi refs"    
            ref = refs[0].strip()
            put_ref_backQ = False
            if ref in note_ref_metadata:
                put_ref_backQ = True
                hardcitationQ, bibblock = note_ref_metadata[ref]
                put_back_keys = put_back_keys|set([ref])
                if ref_count[ref]>1 and (not hardcitationQ):
                    logging.info(f"key {ref} skip, dual to many counts and its not a hardcitation")
                    continue # only when it is not type math and ref > 1 case, we dont insect note into contextf    
                #assert len(texts) == 1, f"Only single citation replacement is supported per cite element.{ref} appear more than once"
                logging.info(f"put back {ref}")
        ## we then replace the entire <cite> to <bibblock> if put_ref_backQ is True
        if put_ref_backQ:
            assert len(all_refs_of_one_cite)==1, "why a cite has multi refs"
            cite.replace_with(bibblock)         
        for key in put_back_keys: ### [Question]: why we should delete the keys
            del note_ref_metadata[key]
    return soup, put_back_keys

def remove_figures_record_the_labels(soup: BeautifulSoup):
    """
        A figure example looks like
        <figure id="S2.F2" class="ltx_figure">
            <div class="ltx_flex_figure">
                <div class="ltx_flex_cell ltx_flex_size_2">
                    <figure id="S2.F2.5" class="ltx_figure ltx_figure_panel ltx_minipage ltx_align_middle" style="width:203.8pt;">
                        <img src="x1.png" id="S2.F2.1.g1" class="ltx_graphics ltx_img_square" width="239" height="239" alt="Refer to caption">
                        <figcaption class="ltx_caption">
                            <span class="ltx_tag ltx_tag_figure">Figure 1:</span>
                            Preliminary CLAS results for $R_{EM}$ of the N$\Delta(1232)$ transition. The curves represent recent models within a constituent quark model including mesons cloud effects <cite class="ltx_cite ltx_citemacro_cite">[<a href="#bib.bib12" title="" class="ltx_ref">12</a>, <a href="#bib.bib13" title="" class="ltx_ref">13</a>]</cite>, and a chiral quark soliton model <cite class="ltx_cite ltx_citemacro_cite">[<a href="#bib.bib14" title="" class="ltx_ref">14</a>]</cite>, respectively
                        </figcaption>
                    </figure>
                </div>
                <div class="ltx_flex_cell ltx_flex_size_2">
                    <figure id="S2.F2.10" class="ltx_figure ltx_figure_panel ltx_minipage ltx_align_middle" style="width:203.8pt;">
                        <img src="x2.png" id="S2.F2.6.g1" class="ltx_graphics ltx_img_square" width="239" height="239" alt="Refer to caption">
                        <figcaption class="ltx_caption">
                            <span class="ltx_tag ltx_tag_figure">Figure 2: </span>
                            Preliminary CLAS results for $R_{SM}$ of the N$\Delta(1232)$
                            transition. Same models as in Figure <a href="#S2.F2" title="Figure 2 ‣ 2 THE QUADRUPOLE TRANSITION TO THE Δ⁢(1232) ‣ Probing the Structure of Nucleons in the Resonance region with CLAS at Jefferson Lab" class="ltx_ref"><span class="ltx_text ltx_ref_tag">2</span></a>.
                        </figcaption>
                    </figure>
                </div>
            </div>
        </figure>
    """

    ### firstly, we located the deepest <figure> tag that contain the <figcaption>
    ### then, we will collect the tag (which will used in cite) and remove the whole figures 
    ## find whole the figures that has nest caption
    def is_caption_figure(tag):
        return tag.name == 'figure' and tag.find('figcaption', recursive=False) is not None
    
    primary_labels = {}
    primary_metadata = {}

    for i, caption_figure in enumerate(soup.find_all(is_caption_figure)):
        captions = caption_figure.find_all('figcaption')
        assert len(captions) == 1, f"Why this element {caption_figure} has multiple caption??"
        caption  = captions[0]
        tags     = caption.find_all(class_='ltx_tag_figure')
        assert len(tags) == 1, f"Why this element {caption} has multiple tags??"
        tag = tags[0].text.strip()
        tag = tag if tag else i+1

        label  = caption_figure.get('id') ## ==> S2.F2.10
        assert label is not None and len(label.strip())> 0, f"why this figure {caption_figure} has no label"
        primary_labels[label]   = tag
        primary_metadata[label] = copy.deepcopy(caption)
        
    ### then we find out whole the figure and record their figure id like
    #### - S2.F2
    #### -- S2.F2.5
    #### -- S2.F2.10
    ### this is for the case the cite will goes to the main figure rather than the subfigure
    def is_main_figure(tag):
        return (tag.name == 'figure') and ('ltx_figure' in tag.get('class', [])) and ('ltx_figure_panel' not in tag.get('class', []))
    for i, main_figure in enumerate(soup.find_all(is_main_figure)):
        main_figure.decompose()
    return primary_labels, primary_metadata

def remove_tables_record_the_labels(soup: BeautifulSoup):
    """
 
    """

    def is_caption_table(tag):
        return tag.name == 'table' and tag.find('tabcaption', recursive=False) is not None
    
    primary_labels = {}
    primary_metadata = {}

    for i, caption_table in enumerate(soup.find_all(is_caption_table)):
        captions = caption_table.find_all('tabcaption')
        assert len(captions) == 1, f"Why this element {caption_table} has multiple caption??"
        caption  = captions[0]
        tags     = caption.find_all(class_='ltx_tag_table')
        assert len(tags) == 1, f"Why this element {caption} has multiple tags??"
        tag = tags[0].text.strip()
        tag = tag if tag else i+1

        label  = caption_table.get('id') ## ==> S2.F2.10
        assert label is None or len(label.strip()) == 0, f"why this table {caption_table} has no label"
        primary_labels[label]   = tag
        primary_metadata[label] = copy.deepcopy(caption)
        
    ### then we find out whole the table and record their table id like
    #### - S2.F2
    #### -- S2.F2.5
    #### -- S2.F2.10
    ### this is for the case the cite will goes to the main table rather than the subtable
    def is_main_table(tag):
        return tag.name == 'table' and 'ltx_table' in tag.get('class', []) and 'ltx_table_panel' not in tag.get('class', [])
    for i, main_table in enumerate(soup.find_all(is_caption_table)):
        main_table.decopose()
    return primary_labels, primary_metadata

def revert_the_block_equation_into_latex(soup:BeautifulSoup):
    """
    Basicly, we dont need recard the tag of the equation since we will put it in the full content
        - [QUESTION] how to let mode ref back to the correct equation if it is a plain content
    """
    primary_labels   = {}
    def is_block_equation(tag):
        return tag.name == 'table' and 'ltx_equation' in tag.get('class', []) 
    for i, block_equation in enumerate(soup.find_all(is_block_equation)):
        tags = block_equation.find_all(class_='ltx_tag_equation')
        label= None
        if len(tags) > 0:
            assert len(tags) == 1, f"Why this element {block_equation.prettify()} has multiple tags??"
            tag = tags[0].text.strip()
            tag = tag if tag else i+1
            label  = block_equation.get('id') ## ==> S2.F2.10
            assert label is not None and len(label.strip()) > 0, f"why this figure {block_equation} has no label"
            primary_labels[label]   = tag
        whole_maths_here = block_equation.find_all('math')
        assert len(whole_maths_here) > 0, f"Why this element {block_equation.prettify()} has no maths?? "
        assert len(whole_maths_here) == 1, f"Why this element {block_equation.prettify()} has multiple maths??"
        math = whole_maths_here[0]
        label_string = f"id={label}" if label else ""
        block_equation.replace_with(BeautifulSoup(f"""<p class="ltx_p" {label_string}>$$\n{better_latex_math_code(math.get('alttext'))}\n$$\n</p>""").p)   
        #block_equation.replace_with(f"$$\n{math.get('alttext').strip()}\n$$\n")
    return primary_labels
        
        
def revert_the_block_equationgroup_into_latex(soup:BeautifulSoup):
    """
    Equationgroup for format block equation like 
    <p class="ltx_p">
     $$ \phi^{AS_{a},AI_{a^{\prime}}}=  2\langle k_{A}\rangle\rho^{AS_{a}}\rho^{AI_{a^{\prime}}} \\
        \phi^{AS_{a},UI_{a^{\prime}}}= \langle k_{A}\rangle\rho^{AS_{a}}\rho^{UI_{a^{\prime}}}, \\
        \phi^{US_{a},AI_{a^{\prime}}}= \langle k_{A}\rangle\rho^{US_{a}}\rho^{AI_{a^{\prime}}}. $$
    
    """
    primary_labels   = {}

    def is_equationgroup(tag):
        return tag.name == 'table' and 'ltx_equationgroup' in tag.get('class', []) 
    
    for  equationgroup in soup.find_all(is_equationgroup):
        if equationgroup.get('id',None):
            father_label = equationgroup.get('id',None)
            primary_labels[father_label]=father_label
        mathlatex_line_by_line = []
        for i,block_equation in enumerate(equationgroup.find_all(class_='ltx_eqn_row')):
            tags = block_equation.find_all(class_='ltx_tag_equation')
            label= None
            if len(tags) > 0:
                assert len(tags) == 1, f"Why this element {block_equation.prettify()} has multiple tags??"
                tag = tags[0].text.strip()
                assert len(tag)>0
                label  = block_equation.get('id') or equationgroup.get('id') ## ==> S2.F2.10 ==> if sub equation dont have we use its father, it usually becasue it is a single row equationgroup
                assert label is not None and len(label.strip()) > 0, f"why this figure {block_equation.prettify()} has no label"
                primary_labels[label]   = tag
            ## for one <tr> row , there are mulitiple latex code 
            ## when using equation group, long math will be placed into different layout like left - right
            whole_maths_here = block_equation.find_all('math')
            if len(whole_maths_here)==0 and i==0:continue
            assert len(whole_maths_here) > 0, f"Why this element {block_equation.prettify()} at has no maths?? "
            #assert len(whole_maths_here) == 1, f"Why this element {block_equation.prettify()} has multiple maths??"
            math_latex = " ".join(better_latex_math_code(math.get('alttext')) for math in whole_maths_here)

            if len(tags)>0 or len(mathlatex_line_by_line)==0:
                mathlatex_line_by_line.append([tags,label, math_latex])
            else:
                mathlatex_line_by_line[-1][-1] +=r' \\ '+ math_latex
        
        mathlatex = "\n".join([f"""<p class="ltx_p" id={label}>$$\n{math_latex}\n$$\n</p>""" for tag, label, math_latex in mathlatex_line_by_line])
        newtag  = BeautifulSoup(f"<div> {mathlatex} </div>")
        equationgroup.replace_with(newtag.div)
            #block_equation.replace_with(BeautifulSoup(f"""<p class="ltx_p">$$\n{math_latex}\n$$\n<p>""").p)
            #block_equation.replace_with(f"$$\n{math.get('alttext').strip()}\n$$\n")        
    return primary_labels

def revert_all_the_math_to_latex(soup:BeautifulSoup):
    for math in soup.find_all('math'):
        math.replace_with(f" ${math.get('alttext').strip()}$ ")
        
def shrink_brackets(input_string):
    # Use a regular expression to replace all occurrences of one or more '[' with a single '['
    input_string = re.sub(r'\[+', '[', input_string)
    input_string = re.sub(r'\]+', ']', input_string)
    return input_string

def recovery_whole_citation_simple(soup: BeautifulSoup):
    """
    A typical citation 
      <cite class="ltx_cite ltx_citemacro_cite">
        <a href="#bib.bib4" title="" class="ltx_ref">fffng </a>; 
        <a href="#bib.bib1" title="" class="ltx_ref">bansal</a>
      </cite>.

    OR
        <cite class="ltx_cite ltx_citemacro_cite">[
            <a href="#bib.bib13" title="" class="ltx_ref">13</a>, 
            <a href="#bib.bib12" title="" class="ltx_ref">12</a>]
        </cite>.
    """
    for cite in soup.find_all(class_='ltx_citemacro_cite'):
        for ref in cite.find_all(class_='ltx_ref'):
            ref = cite.find(class_='ltx_ref')
            ref.replace_with(f"[{ref.text}]")
        cite.replace_with(f" {shrink_brackets(cite.text.replace('],[', ','))}")

    for ref in soup.find_all(class_='ltx_ref'):
        ref.replace_with(f" {ref.text}")        

def beautify_sentence(soup: BeautifulSoup):
    for p in soup.find_all('p'):
        new_content = []
        for element in p.contents:
            if isinstance(element, NavigableString):
                new_content.append(better_latex_sentense_string(str(element)))
            else:
                new_content.append(str(element))
        p.clear()
        p.append(' '.join(new_content))

def beautify_section_title(soup: BeautifulSoup):
    for level in [1,2,3,4,5,6,7]:
        for h in soup.find_all(f'h{level}'):
            h.replace_with("\n\n"+"#"*level+f" {better_latex_sentense_string(h.text)}")
            
def deal_with_itermize(soup):
    for ul in soup.find_all('ul',class_='ltx_itemize'):
        for li in ul.find_all('li'):
            li.replace_with("- "+f"{better_latex_sentense_string(li.text)}")
            
def discard_para(soup):
    for div in soup.find_all('div',class_='ltx_para'):
        div.replace_with(f"\n{div.text.strip()}\n")
        
def discard_section(soup: BeautifulSoup):
    for section in soup.find_all('section',class_='ltx_section'):
        section.replace_with(f"{section.text.strip()}")

def discard_text_format_in_sentense(soup: BeautifulSoup):
    def is_italic(tag):
        return tag.name == 'span' and 'ltx_text' in tag.get('class', []) and 'ltx_font_italic' in tag.get('class', [])
    for italic_text in soup.find_all(is_italic):
        italic_text.replace_with(f"*{better_latex_sentense_string(italic_text.text.strip('*'))}*")
    def is_bold(tag):
        return tag.name == 'span' and 'ltx_text' in tag.get('class', []) and 'ltx_font_bold' in tag.get('class', [])
    for bold_text in soup.find_all(is_italic):
        bold_text.replace_with(f"**{better_latex_sentense_string(bold_text.text.strip('*'))}**")

def remove_and_collect(soup: BeautifulSoup, name):
    element = soup.find(class_=name)
    if element:
        obj = copy.deepcopy(element)
        element.decompose()
        return obj

def remove_and_collect_abstract(soup: BeautifulSoup):
    abstract = soup.find(class_='ltx_abstract')
    if abstract:
        obj = copy.deepcopy(abstract)
        abstract.decompose()
        return obj

    
def collect_appendix_and_remove(soup):
    whole_sections = []
    for section in soup.find_all(class_='ltx_appendix'):
        whole_sections.append(section_to_json(section))
        section.decompose()
    return whole_sections
    
def collect_sections_to_content(soup):
    whole_normal_sections = soup.find_all(class_='ltx_section')
    if len(whole_normal_sections) == 0:
        logging.warn(f'this html doesnt have ltx_section, thus we use whole para directly, make sure no appendix in')
        assert len(soup.find_all('section')) ==0, f"why the html wont have ltx section but have another section type, please check"
        whole_normal_sections = [soup]
    whole_sections = []
    for section in whole_normal_sections:
        whole_sections.append(section_to_json(section))
    return whole_sections

def section_to_json(soup):
    '''
    <h2 class="ltx_title ltx_title_section"> 
        <span class="ltx_tag  ltx_tag_section"> 1 </span> INTRODUCTION 
    </h2>
    <div class="ltx_para" id="S1.p1">
    </div>
    '''
    section = {'title': None, 'paragraph' : []}
    title = soup.find(class_='ltx_title_section')
    if title: section['title']= better_latex_sentense_string(title.text)
    for para in soup.find_all(class_='ltx_para'):
        section['paragraph'].append(para_to_json(para))
    return section

def para_to_json(soup):
    sentenses = []
    for p in soup.find_all(['p','ul']):
        sentenses.append(better_latex_sentense_string(p.text))
    return sentenses

def collect_abstract(soup):
    revert_the_block_equation_into_latex(soup)
    revert_all_the_math_to_latex(soup)
    recovery_whole_citation_simple(soup)
    discard_text_format_in_sentense(soup)
    beautify_sentence(soup)
    
    content = []
    for p in soup.find_all('p'):
        content.append(p.text)
    return "\n".join(content)

def collect_author(soup):
    """
    We only take the name 
    <div class="ltx_authors">
        <span class="ltx_creator ltx_role_author">
        <span class="ltx_personname">Chao-Ran Cai </span>
        <span class="ltx_author_notes">
        <span class="ltx_contact ltx_role_affiliation">School of Physics, Northwest University,Xi’an 710127, China</span>
        <span class="ltx_contact ltx_role_affiliation">Shaanxi Key Laboratory for Theoretical Physics Frontiers, Xi’an 710127, China </span></span></span>
        <span class="ltx_author_before"></span><span class="ltx_creator ltx_role_author">
        <span class="ltx_personname">Yuan-Yuan Nie </span><span class="ltx_author_notes">
        <span class="ltx_contact ltx_role_affiliation">School of Physics, Northwest University, Xi’an 710127, China </span></span></span>
        <span class="ltx_author_before"></span><span class="ltx_creator ltx_role_author">
        <span class="ltx_personname">Petter Holme </span><span class="ltx_author_notes">
        <span class="ltx_contact ltx_role_email"><a href="mailto:petter.holme@aalto.fi">petter.holme@aalto.fi</a> </span>
        <span class="ltx_contact ltx_role_affiliation">Department of Computer Science, Aalto University, Espoo, Finland </span>
        <span class="ltx_contact ltx_role_affiliation">Center for Computational Social Science, Kobe University, Kobe, Japan </span></span></span>
    </div>
    """
    if soup is None: return
    revert_all_the_math_to_latex(soup)
    authors = []
    for author_name in soup.find_all(class_='ltx_personname'):
        authors.append(better_latex_sentense_string(author_name.text))
    return authors

def collect_acknowledgements(soup):
    """
    <div class="ltx_acknowledgements">
        <h6 class="ltx_title ltx_title_acknowledgements">Acknowledgements.</h6>
        This work was supported by the Shaanxi Fundamental Science Research Project for Mathematics and
        Physics (Grant No. 22JSQ003). PH was supported by JSPS KAKENHI Grant Number JP 21H04595.
    <div>
    
    """
    if soup is None: return
    ### remove the title
    for title in soup.find_all(class_='ltx_title'):title.decompose()
    revert_all_the_math_to_latex(soup)
    return better_latex_sentense_string(soup.text)

def cleanup_html(soup, whole_ref_to_labels,paper_id,refs_that_wont_recovery=[]):
    revert_the_block_equationgroup_into_latex(soup)
    revert_the_block_equation_into_latex(soup)
    revert_all_the_math_to_latex(soup)
    #recovery_whole_citation_simple(soup)
    recovery_whole_citation_complete(soup,whole_ref_to_labels, paper_id,refs_that_wont_recovery)
    discard_text_format_in_sentense(soup)
    beautify_sentence(soup)
    deal_with_itermize(soup)
    #beautify_section_title(soup)
    #tree = replace_item_block_with_markdown_format(tree)

def cleanup_reference_string(soup, whole_ref_to_labels,paper_id, refs_that_wont_recovery):
    cleanup_html(soup, whole_ref_to_labels, paper_id, refs_that_wont_recovery = refs_that_wont_recovery)
    string = soup.text.replace("[[[Notice:","").replace("]]]","")
    return better_latex_sentense_string(string)

def recovery_citation_in_sentense(cite: BeautifulSoup, labels_reference: Dict[str, str], paper_id: str, refs_that_wont_recovery=[]):
    refs = []
    for ref in cite.find_all(class_='ltx_ref'):
        ref_key = ref['href']
        if ref_key:
            refs.append(ref_key.strip().lstrip('#'))
    
    refs = [ref for ref in refs if ref not in refs_that_wont_recovery]
    
    label_list = []
    for ref in refs:
        reflabels = labels_reference[ref]
        if len(reflabels) > 1:
            logging.info(f"multiple label detected: {reflabels}, we will use the first one")
        ref_type, label = reflabels[0]
        label_list.append(label.strip("[](){}"))
    if len(label_list) ==0:
        cite.decompose()
        return 
    
    label = ",".join(label_list)    
    label = "[" + label + "]"
    next_string = cite.next_sibling 
    prev_string = cite.previous_sibling
    
    right       = next_string.strip() if next_string else ""
    left        = prev_string.strip() if prev_string else ""
    left, right = discard_brackets(left,right)
    
    #leftbrace   = "" if left and left.strip()[-1] in ['[','(','{']  else "["
    #rightbrace  = "" if right and right.strip()[0] in [']',')','}'] else "]"
    #label = leftbrace + label + rightbrace
    
    if left:
        left, label, isactivated = go_ahead_and_add_label(left, label, paper_id)      
    else:
        left = ""
    label = format_the_smart_citation(left, label, right, paper_id)
    
    if prev_string:prev_string.replace_with(left)
    cite.replace_with(label)
    if next_string:next_string.replace_with(right)

def recovery_ref_in_sentense(ref: BeautifulSoup, labels_reference: Dict[str, str], paper_id: str,refs_that_wont_recovery=[]):
    ref_key = ref.get('href', None)
    if not ref_key:
        logging.warning(f"""ref of element dont have ref??? See {ref.prettify()}""")
        return
    ref_key = ref_key.strip().lstrip('#')
    if ref_key not in labels_reference:
        logging.warning(f"""{ref_key} not in labels_reference, please check.""")
        #ref.replace_with(f" {ref.text}")       #Degenerate to simple mode  
        return
    
    reflabels = labels_reference[ref_key]
    if len(reflabels) > 1:
        logging.info(f'multiple label detected for {labelref} => {reflabels}, we will use the first one')
    
    ref_type, label = reflabels[0]
    next_string = ref.next_sibling 
    prev_string = ref.previous_sibling
    
    right       = next_string.strip() if next_string else ""
    left        = prev_string.strip() if prev_string else ""
    left, right = discard_brackets(left,right)
    
    # if left:
    #     left, label, isactivated = go_ahead_and_add_label(left, label, paper_id)      
    # else:
    #     left = ""
    # if not isactivated:
    #     if ref_type.lower() in ['url']:
    #         label = label
    #     elif ref_type.lower() not in ['equation','formula']:
    #         label = f" (See [{ref_type}.{label} of {paper_id}]) "
    #     else:
    #         label = f"[{ref_type}.{label} of {paper_id}]"
    # ref.replace_with(label)
    
    if left:
        left, label_new, isactivated = go_ahead_and_add_label(left, label, paper_id)      
    else:
        left = ""
    if not isactivated:
        if ref_type.lower() in ['url']:
            label = label
        elif ref_type.lower() not in ['equation', 'formula']:
            label = f" (See [{ref_type}.{label} of {paper_id}]) "
        else:
            label = f"[{ref_type}.{label} of {paper_id}]"    
    else:
        label = label_new

    if prev_string:prev_string.replace_with(left)
    ref.replace_with(label)
    if next_string:next_string.replace_with(right)


def recovery_whole_citation_complete(soup: BeautifulSoup,whole_ref_to_labels, paper_id,refs_that_wont_recovery=[]):
    """
        firstly deal with <ref> in <cite>
        then deal with <ref> for figure. math. table, and so on
    """
    for cite in soup.find_all(class_='ltx_citemacro_cite'):
        recovery_citation_in_sentense(cite, whole_ref_to_labels, paper_id,refs_that_wont_recovery=refs_that_wont_recovery)

    for ref in soup.find_all(class_='ltx_ref'):
        recovery_ref_in_sentense(ref, whole_ref_to_labels, paper_id,refs_that_wont_recovery=refs_that_wont_recovery)

def format_the_smart_citation(left, label, right, paper_id):
    citation_content = label
    # Determine if label is enclosed in angle brackets and format accordingly
    if label.startswith('<') and label.endswith('>'):
        citation_content = f'[{label[1:-1]}]'
    else:
        citation_content = f'[Ref.{label} of {paper_id}]'
    
    # Determine position and format output
    is_start_of_string = not left.strip() or left.strip()[-1] in {'.', '!', '?', ';'}
    end_mark = not right.strip() or right.strip()[0].isupper()
    is_after_position = False
    if left:
        is_after_position = left.split()[-1] in prepositions
    
    if is_start_of_string or is_after_position:
        output = citation_content
    else:
        output = f'(See {citation_content})'
    
    if end_mark:
        output += '.'
    
    return output

def remove_tags_and_record_all_labels(soup):
    ### please make sure you remove the figure, table, equation 
    otherslabels={}
    for tag in soup.find_all(class_='ltx_tag'):
        label_type = [_type for _type in tag.get('class', []) if _type.startswith('ltx_tag_')]
        if len(label_type)>0:
            assert len(label_type)==1
            label_type = label_type[0].replace('ltx_tag_','').capitalize()
        else:
            label_type = 'Other'
        if label_type not in otherslabels:otherslabels[label_type]={}
        parent = tag
        max_deepth=10
        now_deep = 0
        while not parent.get('id',None):
            parent = parent.parent
            now_deep+=1
            if now_deep>max_deepth:break
        label = parent.get('id',None)
        if label:
            otherslabels[label_type][label.strip()] = tag.text.strip()
    return otherslabels

# Main

In [2]:
args = HTMLtoJsonConfig(root_path="")
args.filte_out_note = not args.passNote

output_dir    = 'debug'
tmp_html_path = '/nvme/zhangtianning/datasets/ar5iv/no-problem/9906/hep-ph9906398.html'

use_count_type_ref  = not args.use_origin_ref_number
reterive_result_mode=args.reterive_result_mode
_paper_id =os.path.basename(tmp_html_path.replace('.html',''))
paper_filter = ContextFilter(_paper_id)
logging.getLogger().addFilter(paper_filter)
paper_id = f"ArXiv.{_paper_id}"
soup_whole = BeautifulSoup(open(tmp_html_path),'html.parser')
soup = soup_whole.find('article')


ReferenceDir= os.path.join(output_dir, "Reference")
author  = remove_and_collect(soup, 'ltx_authors')
abstract= remove_and_collect(soup, 'ltx_abstract')
acknowledgements= remove_and_collect(soup, 'ltx_acknowledgements')

discard_note(soup)
ref_count = retrieve_all_cite(soup)
new_soup = deepcopy(soup)
new_soup,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(new_soup,ref_count, args)
if len(note_ref_metadata)>5:
    for key,val in note_ref_metadata.items():
        logging.info(f"{key} ==> [ {better_latex_sentense_string(' '.join(val[1].text))} ]")
    checkTooManyNote(f'the note_ref_metadata num={len(note_ref_metadata) } is too much , please check the file {tmp_html_path}')
    logging.warning('WARNING:Too Many note, we roll back to no note mode')
    args.passNote = True
    soup,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(soup,ref_count,args)
else:
    soup= new_soup

reference_labels, reference_labels_not_in_context = divide_the_dict_into_two_part_by_keys(reference_labels, ref_count)
note_ref_labels, note_ref_labels_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_labels,ref_count)
bibitem_ref_metadata, bibitem_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(bibitem_ref_metadata,ref_count)
note_ref_metadata, note_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_metadata,ref_count)


put_back_keys = put_note_string_back_into_each_sentence(soup,ref_count,note_ref_metadata)
## since we put those key back into content, we never need those key anymore and wont save them in the reference.txt
for key in put_back_keys:
    del note_ref_labels[key]
    del note_ref_metadata[key]
# notice, after this line, the key in note_ref_metadata and note_ref_labels is different

figures_labels, figures_metadata = remove_figures_record_the_labels(soup)
tables_labels, tables_metadata   = remove_tables_record_the_labels(soup)
floats_labels, floats_metadata   = remove_floats_record_the_labels(soup)
equation_group_labels =revert_the_block_equationgroup_into_latex(soup)
equation_labels,extra_table_label, extra_table_metadata      =revert_the_block_equation_into_latex(soup)
tables_labels = tables_labels|extra_table_label
tables_metadata=tables_metadata|extra_table_metadata
in_content_ref_labels = {
    'Figure':figures_labels,
    'Table':tables_labels,
    'Equation':equation_labels,
    'Equationgroup':equation_group_labels,
    'Floats':floats_labels
}

In [4]:
revert_all_the_math_to_latex(soup)
with open('test.html','w') as f:
    f.write(soup.prettify())

In [7]:
for main_tables,caption_tag in [(soup.find_all(is_span_table),'ltx_caption')]:
        for main_table in main_tables:
            whole_captions = main_table.find_all(class_=caption_tag) if caption_tag.startswith('ltx') else main_table.find_all(caption_tag)

In [3]:
labels               = collect_tags_and_record_all_labels(soup)## like section and so one
extra_figure_label, extra_figure_metadata, extra_table_label,extra_table_metadata = check_no_figure_and_table_left(soup)
assert len(set(in_content_ref_labels)&set(labels))==0, f"the remain tag should not include those collect before. \n collect before:{in_content_ref_labels.keys()}\n now:{labels.keys()}"

In [6]:
revert_all_the_math_to_latex(soup)
with open('test.html','w') as f:
    f.write(soup.prettify())

In [5]:
all_citation_keys = set(ref_count)
all_reference_keys= (set(reference_labels)|
                     set(note_ref_labels)|
                     set(figures_labels)|
                     set(tables_labels)|
                     set(equation_labels)|
                     set(equation_group_labels)|
                     set(equation_group_labels)|
                     set(floats_labels))

for val_pool in labels.values():
    all_reference_keys = all_reference_keys | set(val_pool)
missing_citation = all_citation_keys - all_reference_keys
missing_citation_labels = {missing_citation_label:f'MissingCite_{i}' for i,missing_citation_label in enumerate(missing_citation)}


#assert len(bibitem_ref_metadata)>0, f"Error: this file [{tmp_xml_path}] donts have bib???"

if reterive_result_mode:
    assert os.path.exists(os.path.join(ReferenceDir,'reference.keys.done'))
    assert os.path.getsize(os.path.join(ReferenceDir,'reference.txt')) == 0, "if you want to inject the reterive result, please make sure all the element is reterived"
    with open(os.path.join(ReferenceDir,'reference.keys.done'),'r') as f:
        reference_keys = [t.strip() for t in f]
    with open(os.path.join(ReferenceDir,'reference.es_retrived_citation.json.done'),'r') as f:
        reference_reterives = json.load(f)
    assert len(reference_keys) == len(reference_reterives), "the reterive result should have the same length as the keys"
    new_label_mapping = {}
    for key, reterive_result in zip(reference_keys,reference_reterives):
        if key not in new_label_mapping:new_label_mapping[key] = []
        new_label_mapping[key].append(get_unique_id_from_reterive_result(reterive_result))
    for key in new_label_mapping.keys():
        new_label_mapping[key] = "<"+ ",".join(new_label_mapping[key]) + ">"
    reference_labels = new_label_mapping


whole_ref_to_labels = collect_whole_reference(in_content_ref_labels|
                                              {'Reference':reference_labels,'Missing':missing_citation_labels}|
                                              labels, 
                                              use_count_type_ref=use_count_type_ref)

lack_ref = list(set(ref_count) - (set(all_reference_keys)|set(whole_ref_to_labels)))
if len(lack_ref)>0:
    logging.info(f'you have {len(lack_ref)} ref lacks, such as {lack_ref[:4]}, please check the file {tmp_html_path}')
    raise MisMatchRefError

## now, the left note metadata is those string looks like a citation, and we will put them back into the bibitem information
for remain_key, remain_val in note_ref_metadata.items():

    reference_labels[remain_key]=note_ref_labels[remain_key]
    string = cleanup_reference_string(remain_val[1], whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
    bibitem_ref_metadata[remain_key]=better_latex_sentense_string(string)


whole_ref_to_labels = collect_whole_reference(in_content_ref_labels|
                                              {'Reference':reference_labels,'Missing':missing_citation_labels}|
                                              labels, 
                                              use_count_type_ref=use_count_type_ref)

cleanup_html(soup, whole_ref_to_labels,paper_id,refs_that_wont_recovery=[])


for remain_key, remain_val in note_ref_metadata_not_in_context.items():
        string = cleanup_reference_string(remain_val[1], whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
        note_ref_metadata_not_in_context[remain_key]=better_latex_sentense_string(string)
        ## do this again since we modify the bibitem_ref_metadata

for metadatapool in [figures_metadata, tables_metadata, floats_metadata]:
    for remain_key, remain_val in metadatapool.items():
        string = cleanup_reference_string(remain_val, whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
        metadatapool[remain_key]=better_latex_sentense_string(string)


whole_metadata = {'figures_metadata':figures_metadata,
                  'tables_metadata':tables_metadata,
                  'floats_metadata':floats_metadata,
                  'bibitem_ref_metadata':bibitem_ref_metadata,}



In [6]:
content_soup = copy.deepcopy(soup)
appendix_content = collect_specific_section_and_remove(content_soup,name='ltx_appendix')
index_content    =  collect_specific_section_and_remove(content_soup,name='ltx_index')
sections_content = collect_sections_to_content(content_soup)
assert len(content_soup.find_all('section')) ==0, f"why the html wont have ltx section but have another section type, please check"

AssertionError: why the html wont have ltx section but have another section type, please check

In [7]:

with open('test.html','w') as f:
    f.write(content_soup.prettify())

In [7]:
with open('test.html','w') as f:
    f.write(content_soup.prettify())

In [13]:
output_dict = {'abstract':collect_abstract(abstract),
               'acknowledge':collect_acknowledgements(acknowledgements),
               'author': collect_author(author),
               'appendix':appendix_content,
               'sections':sections_content,
               'metadata':whole_metadata,
               'paper_id':paper_id,
               'whole_ref_to_labels':whole_ref_to_labels,
               'missing_citation_labels':missing_citation_labels}

Paper ID: 0806.1883 - S2.E11 not in labels_reference, please check.


NotImplementedError: MUST raise here, as it will go infinity loop

In [8]:
whole_ref_to_labels

{'bib.bib1': [('Reference', '0')],
 'bib.bib2': [('Reference', '1')],
 'bib.bib3': [('Reference', '2')],
 'bib.bib4': [('Reference', '3')],
 'bib.bib5': [('Reference', '4')],
 'bib.bib6': [('Reference', '5')],
 'bib.bib7': [('Reference', '6')],
 'bib.bib8': [('Reference', '7')],
 'bib.bib9': [('Reference', '8')],
 'bib.bib10': [('Reference', '9')],
 'bib.bib11': [('Reference', '10')],
 'bib.bib12': [('Reference', '11')],
 'bib.bib13': [('Reference', '12')],
 'bib.bib14': [('Reference', '13')],
 'bib.bib15': [('Reference', '14')],
 'bib.bib16': [('Reference', '15')],
 'bib.bib17': [('Reference', '16')],
 'bib.bib18': [('Reference', '17')],
 'bib.bib19': [('Reference', '18')],
 'bib.bib20': [('Reference', '19')],
 'bib.bib21': [('Reference', '20')],
 'bib.bib22': [('Reference', '21')],
 'bib.bib23': [('Reference', '22')],
 'bib.bib24': [('Reference', '23')],
 'bib.bib25': [('Reference', '24')],
 'bib.bib26': [('Reference', '25')],
 'bib.bib27': [('Reference', '26')],
 'bib.bib28': [('Ref

In [5]:
output_dict = {'abstract':collect_abstract(abstract),
               'acknowledge':collect_acknowledgements(acknowledgements),
               'author': collect_author(author),
               'appendix':appendix_content,
               'sections':sections_content,
               'metadata':whole_metadata,
               'paper_id':paper_id,
               'whole_ref_to_labels':whole_ref_to_labels,
               'missing_citation_labels':missing_citation_labels}

In [16]:
revert_all_the_math_to_latex(soup)
with open('test.html','w') as f:
    f.write(soup.prettify())

In [ ]:
put_ref_back_and_clean_format(tree,whole_ref_to_labels,paper_id=paper_id,refs_that_wont_recovery=refs_that_wont_recovery)
tree = beauty_each_sentense(tree)
tree = replace_equation_blocks(tree)
tree = replace_item_block_with_markdown_format(tree)

In [13]:
from bs4 import BeautifulSoup

def html_to_markdown(html):
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table')

    # Extracting all rows from the table
    rows = table.find_all('tr')
    markdown_table = []

    # Process each row
    for row in rows:
        cells = row.find_all('td')
        if not cells:
            continue
        # Extract text from each cell
        extracted_cells = [cell.get_text(strip=True) for cell in cells]
        markdown_table.append(extracted_cells)

    # Build the Markdown table
    markdown_output = ""
    headers = markdown_table[0]
    alignment_row = ["---"] * len(headers)
    
    # Create the header row
    header_row = "| " + " | ".join(headers) + " |"
    markdown_output += header_row + "\n"
    
    # Create the alignment row
    alignment_row = "| " + " | ".join(alignment_row) + " |"
    markdown_output += alignment_row + "\n"
    
    # Add the rest of the data rows
    for data_row in markdown_table[1:]:
        row = "| " + " | ".join(data_row) + " |"
        markdown_output += row + "\n"
        
    return markdown_output

# Example HTML input
html_content = """
<table class="ltx_tabular ltx_markedasmath ltx_align_bottom" id="S5.Ex1.1">
<tr class="ltx_tr" id="S5.Ex1.1.1">
<td class="ltx_td ltx_nopad_l ltx_nopad_r ltx_border_l ltx_border_t" id="S5.Ex1.1.1.1"></td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.1.2">1 Tick</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.1.3">2 Ticks</td>
<td class="ltx_td ltx_nopad_r ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.1.4">4 Ticks</td>
<td class="ltx_td ltx_nopad_r ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.1.5">8 Ticks</td>
<td class="ltx_td ltx_nopad_r ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.1.6">32 Ticks</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_r ltx_border_t" id="S5.Ex1.1.1.7">128 Ticks</td>
</tr>
<tr class="ltx_tr" id="S5.Ex1.1.2">
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.1">USD/JPY</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.2">0.668</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.3">0.743</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.4">0.807</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.5">0.862</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.2.6">0.938</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_r ltx_border_t" id="S5.Ex1.1.2.7">0.98</td>
</tr>
<tr class="ltx_tr" id="S5.Ex1.1.3">
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.1">USD/GBP</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.2">0.596</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.3">0.732</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.4">0.783</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.5">0.843</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_t" id="S5.Ex1.1.3.6">0.949</td>
<td class="ltx_td ltx_align_left ltx_border_l ltx_border_r ltx_border_t" id="S5.Ex1.1.3.7">0.98</td>
</tr>
<tr class="ltx_tr" id="S5.Ex1.1.4">
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.1">GBP/CHF</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.2">0.88</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.3">0.992</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.4">0.954</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.5">0.98</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_t" id="S5.Ex1.1.4.6">0.99</td>
<td class="ltx_td ltx_align_left ltx_border_b ltx_border_l ltx_border_r ltx_border_t" id="S5.Ex1.1.4.7">0.99</td>
</tr>
</table>
"""

print(html_to_markdown(html_content))

|  | 1 Tick | 2 Ticks | 4 Ticks | 8 Ticks | 32 Ticks | 128 Ticks |
| --- | --- | --- | --- | --- | --- | --- |
| USD/JPY | 0.668 | 0.743 | 0.807 | 0.862 | 0.938 | 0.98 |
| USD/GBP | 0.596 | 0.732 | 0.783 | 0.843 | 0.949 | 0.98 |
| GBP/CHF | 0.88 | 0.992 | 0.954 | 0.98 | 0.99 | 0.99 |



In [18]:
re.split(r'(?<!\\) ', "(\\vec{p}, \\vec{n})")

['(\\vec{p},', '\\vec{n})']